In [10]:
import pandas as pd
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer, util

dummy_data = {
    'book_id': [101, 102, 103, 104, 105, 106],
    'title': ['Deep Learning with Python', 'NLP in Action', 'Advanced CNNs', 'Structural Dynamics', 'Intro to Machine Learning', 'Design of Concrete Structures'],
    'author': ['Chollet', 'Lane', 'Goodfellow', 'Smith', 'Ng', 'Nilson'],
    'genre': ['AI', 'AI & NLP', 'AI & Vision', 'Civil Engineering', 'Machine Learning', 'Civil Engineering'],
    'abstract': [
        'Learn neural networks, deep learning, and computer vision using Keras.',
        'A guide to NLP, text generation, and understanding human language.',
        'Advanced computer vision techniques using CNNs for image classification.',
        'Principles of structural dynamics, civil engineering, and concrete design.',
        'Basics of machine learning, algorithms, and data analysis.',
        'Comprehensive guide to designing reinforced concrete structures.'
    ]
}
df_nlp = pd.DataFrame(dummy_data)
df_nlp['soup'] = df_nlp['title'] + " " + df_nlp['author'] + " " + df_nlp['genre'] + " " + df_nlp['abstract']

print("Loading Model & Encoding Books...")
nlp_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

book_embeddings = nlp_model.encode(df_nlp['soup'].tolist(), convert_to_tensor=True)

Loading Model & Encoding Books...


In [11]:
def chatbot_semantic_search(user_message, model, df, corpus_embeddings, top_k=2, similarity_threshold=0.2):
    """
    Takes user's natural language message and returns recommended books.
    """
    query_embedding = model.encode(user_message, convert_to_tensor=True)
    
    cosine_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
    
    top_results = np.argsort(cosine_scores.cpu().numpy())[::-1][:top_k]
    
    final_picks = []
    for idx in top_results:
        score = float(cosine_scores[idx])
        if score >= similarity_threshold:
            final_picks.append({
                "recommended_book_id": int(df.iloc[idx]['book_id']),
                "title": df.iloc[idx]['title'],
                "match_score": round(score * 100, 1)
            })
    
    if not final_picks:
        return {
            "bot_reply": "عذراً، لم أتمكن من العثور على كتب تطابق طلبك بدقة في مكتبتنا حالياً. هل يمكنك توضيح بحثك أكثر؟",
            "recommended_books": []
        }
    else:
        return {
            "bot_reply": "لقد قمت بالبحث في المكتبة، ووجدت هذه الكتب التي تناسب ما تبحث عنه تماماً:",
            "recommended_books": final_picks
        }

In [12]:
print("Test 1")
test_message_1 = "عايز مراجع عن تصميم الخرسانة والمباني"
response_1 = chatbot_semantic_search(test_message_1, nlp_model, df_nlp, book_embeddings)
import json
print(json.dumps(response_1, indent=4, ensure_ascii=False))

print("\n" + "="*50 + "\n")

print("Test 2")
test_message_2 = "لو سمحت انا لسه مبتدئ وعايز اتعلم ذكاء اصطناعي ابدأ منين؟"
response_2 = chatbot_semantic_search(test_message_2, nlp_model, df_nlp, book_embeddings)
print(json.dumps(response_2, indent=4, ensure_ascii=False))

Test 1
{
    "bot_reply": "لقد قمت بالبحث في المكتبة، ووجدت هذه الكتب التي تناسب ما تبحث عنه تماماً:",
    "recommended_books": [
        {
            "recommended_book_id": 106,
            "title": "Design of Concrete Structures",
            "match_score": 67.9
        },
        {
            "recommended_book_id": 104,
            "title": "Structural Dynamics",
            "match_score": 50.5
        }
    ]
}


Test 2
{
    "bot_reply": "لقد قمت بالبحث في المكتبة، ووجدت هذه الكتب التي تناسب ما تبحث عنه تماماً:",
    "recommended_books": [
        {
            "recommended_book_id": 102,
            "title": "NLP in Action",
            "match_score": 40.1
        },
        {
            "recommended_book_id": 101,
            "title": "Deep Learning with Python",
            "match_score": 39.9
        }
    ]
}


In [15]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv(override=True)
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("GEMINI_API_KEY is missing! Please check your .env file.")

genai.configure(api_key=API_KEY)

llm_model = genai.GenerativeModel('gemini-3.1-flash-lite-preview')

def generate_personalized_rag(user_message, search_results, user_history_books=None):
    """
    Personalized RAG Engine: Uses Semantic Search results + User's Reading History.
    """
    if not search_results:
        return "عذراً، لم أجد كتباً تطابق طلبك بدقة. هل يمكنك توضيح بحثك أكثر؟"
    
    context_text = ""
    for book in search_results:
        context_text += f"- اسم الكتاب: [{book['title']}]\n"
        context_text += f"  نسبة تطابق المحتوى: {book['match_score']}%\n"

    history_text = ""
    if user_history_books:
        history_text = "تاريخ الطالب الأكاديمي (الكتب التي قرأها مسبقاً في المكتبة):\n"
        for title in user_history_books:
            history_text += f"- [{title}]\n"
    else:
        history_text = "الطالب جديد وليس لديه تاريخ استعارات سابق."

    prompt = f"""
    أنت المساعد الرقمي الرسمي لمكتبة الجامعة. دورك هو إرشاد الطلاب للمصادر الأكاديمية باحترافية.
    طريقتك رسمية، مهنية، وأكاديمية باللغة العربية الفصحى. يُمنع التودد الزائد أو الألفاظ العامية.
    يُسمح بل يُفضل الاحتفاظ بالمصطلحات التقنية باللغة الإنجليزية كما هي لضمان الدقة.

    معلومات عن الطالب:
    {history_text}

    استفسار الطالب الحالي: "{user_message}"
    
    نتائج البحث المطابقة من فهرس المكتبة للإجابة على سؤاله:
    {context_text}
    
    المطلوب منك:
    1. تحية رسمية عملية.
    2. عرض الكتب المقترحة في نقاط (Bullet points) فقط، مع شرح أكاديمي موجز لكيفية خدمتها لاستفساره.
    3. (التخصيص الذكي): اربط بذكاء بين استفساره الحالي وبين الكتب التي قرأها مسبقاً (إن وُجدت) لتبين له مسار تطوره الأكاديمي.
    4. حماية التنسيق: ضع أي مصطلح أو اسم كتاب إنجليزي بين أقواس مربعة [ ].
    5. خاتمة رسمية موجزة.
    
    قيود صارمة (RAG Constraints):
    - لا تقترح أي كتاب من خارج (نتائج البحث المطابقة) بأي شكل من الأشكال.
    - خلو النص تماماً من أي رموز تعبيرية (Emojis).
    """
    
    try:
        response = llm_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"حدث خطأ في الاتصال: {e}"

student_past_reads = ["Introduction to Machine Learning"] 
current_question = "عايز مرجع يساعدني أتعلم الـ AI"

search_results = chatbot_semantic_search(current_question, nlp_model, df_nlp, book_embeddings)
actual_books = search_results.get("recommended_books", [])

print("يتم الآن صياغة الرد المخصص للطالب...\n")
print("=" * 60)
print(generate_personalized_rag(current_question, actual_books, student_past_reads))

يتم الآن صياغة الرد المخصص للطالب...

تحية طيبة،

بناءً على اطلاعكم السابق على كتاب [Introduction to Machine Learning]، وتماشياً مع رغبتكم في التوسع في مجال الذكاء الاصطناعي [AI]، فقد حدد فهرس المكتبة المصادر الأكاديمية التالية التي تخدم مساركم المعرفي:

*   [Deep Learning with Python]: يعد هذا المرجع الامتداد المنطقي لما اكتسبتموه من مفاهيم في [Machine Learning]. حيث ينتقل بكم من الخوارزميات التقليدية إلى الشبكات العصبية العميقة [Neural Networks]، مما يعزز قدراتكم في بناء وتدريب نماذج الذكاء الاصطناعي باستخدام لغة [Python].
*   [NLP in Action]: يمثل هذا الكتاب تطبيقاً تخصصياً متقدماً في أحد أهم فروع الذكاء الاصطناعي وهو معالجة اللغات الطبيعية [Natural Language Processing]. سيمكنكم هذا المرجع من فهم كيفية تفاعل الآلة مع البيانات النصية، وهو مسار حيوي يكمل خلفيتكم المعرفية المكتسبة مسبقاً.

إن هذه المصادر المختارة تمثل ترابطاً منهجياً يربط بين الأساسيات النظرية التي درستموها سابقاً وبين التطبيقات التقنية المعاصرة في هذا المجال.

نحن في خدمتكم لأي استفسارات بحثية إضافية.

مع التقدير،
مكت

In [14]:
import joblib
import os
import torch

if isinstance(book_embeddings, torch.Tensor):
    embeddings_np = book_embeddings.cpu().numpy()
else:
    embeddings_np = np.array(book_embeddings)
    
joblib.dump(embeddings_np, 'production_models/nlp_book_embeddings.pkl')
joblib.dump(df_nlp, 'production_models/nlp_books_dataframe.pkl')

['production_models/nlp_books_dataframe.pkl']